In [46]:
import numpy as np
import random

# 1. Define the specific chromosomes given in the problem
population_data = [
    [12, 5, 23, 8],
    [2, 21, 18, 3],
    [10, 4, 13, 14],
    [20, 1, 10, 6],
    [1, 4, 13, 19],
    [20, 5, 17, 1]
]

# Convert to numpy array for easier handling
population = np.array(population_data)

# 2. Define the Fitness Function
def calculate_fitness(chromosome):
    # a, b, c, d corresponds to indices 0, 1, 2, 3
    a, b, c, d = chromosome
    val = (a + 2*b + 3*c + 4*d) - 30
    error = abs(val)
    # We add 1 to avoid division by zero. Higher is better.
    return 1.0 / (1.0 + error)

# 3. Roulette Wheel Selection
def roulette_wheel_selection(population, fitness_scores):
    total_fitness = sum(fitness_scores)
    probs = [f / total_fitness for f in fitness_scores]
    
    # Select indices based on probability
    # We select 2 parents
    indices = np.random.choice(len(population), size=2, p=probs, replace=True)
    return population[indices[0]], population[indices[1]]

# 4. Crossover (Single Point)
def crossover(parent1, parent2):
    if random.random() > 0.8: # 80% crossover rate
        point = random.randint(1, len(parent1) - 1)
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        return child1, child2
    return parent1, parent2

# 5. Mutation
def mutate(child):
    if random.random() < 0.1: # 10% mutation rate
        # Pick a random gene and add a small random integer (-1 to 1)
        idx = random.randint(0, len(child) - 1)
        child[idx] = child[idx] + random.randint(-2, 2)
        # Ensure values don't become negative if that's a constraint (optional)
        child[idx] = max(0, child[idx]) 
    return child

# --- Main GA Loop ---
generations = 100  # Keeping it short for demonstration

print("--- Initial Population & Fitness ---")
for i, ind in enumerate(population):
    print(f"Ch{i+1}: {ind}, Fitness: {calculate_fitness(ind):.4f}, Value: {(ind[0]+2*ind[1]+3*ind[2]+4*ind[3])-30}")

for gen in range(generations):
    # Calculate fitness for all
    fitness_scores = [calculate_fitness(ind) for ind in population]
    
    new_population = []
    
    # Create next generation
    while len(new_population) < len(population):
        # Selection
        p1, p2 = roulette_wheel_selection(population, fitness_scores)
        
        # Crossover
        c1, c2 = crossover(p1, p2)
        
        # Mutation
        c1 = mutate(c1)
        c2 = mutate(c2)
        
        new_population.extend([c1, c2])
    
    population = np.array(new_population[:len(population_data)]) # Keep population size constant

    # Check for perfect solution
    best_fitness = max(fitness_scores)
    # print(f"Gen {gen+1}: Best Fitness: {best_fitness:.4f}")
    if best_fitness == 1.0:
        print(f"\nSolution Found in Gen {gen}!")
        break

# Final Results
best_idx = np.argmax([calculate_fitness(ind) for ind in population])
best_sol = population[best_idx]
val = (best_sol[0] + 2*best_sol[1] + 3*best_sol[2] + 4*best_sol[3]) - 30

print("\n--- Final Best Solution ---")
print(f"Chromosome: {best_sol}")
print(f"Function Value (Target 0): {val}")

--- Initial Population & Fitness ---
Ch1: [12  5 23  8], Fitness: 0.0106, Value: 93
Ch2: [ 2 21 18  3], Fitness: 0.0123, Value: 80
Ch3: [10  4 13 14], Fitness: 0.0119, Value: 83
Ch4: [20  1 10  6], Fitness: 0.0213, Value: 46
Ch5: [ 1  4 13 19], Fitness: 0.0105, Value: 94
Ch6: [20  5 17  1], Fitness: 0.0179, Value: 55

Solution Found in Gen 99!

--- Final Best Solution ---
Chromosome: [2 2 8 0]
Function Value (Target 0): 0


In [42]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import random

# 1. Load the "Rare" Medical Dataset (Diabetes)
# This dataset measures 10 baseline variables (age, sex, bmi, etc.) 
# and the target is a quantitative measure of disease progression.
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

print(f"Dataset Loaded: Diabetes (Regression Task)")
print(f"Total Features: {len(feature_names)} ({feature_names})")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# GA Parameters
POPULATION_SIZE = 10
GENERATIONS = 10     # Increased slightly to see convergence
MUTATION_RATE = 0.1
NUM_FEATURES = X.shape[1]

# 2. Initialize Population
# Binary mask: 1 = Include feature, 0 = Exclude feature
population = np.random.randint(2, size=(POPULATION_SIZE, NUM_FEATURES))

# Safety check: Ensure no chromosome is all zeros
for i in range(POPULATION_SIZE):
    if np.sum(population[i]) == 0:
        population[i][random.randint(0, NUM_FEATURES-1)] = 1

# 3. Fitness Function (Adapted for Regression)
def calculate_fitness_fs(mask):
    indices = [i for i, x in enumerate(mask) if x == 1]
    
    # Penalize empty selection heavily
    if len(indices) == 0:
        return 0.0
    
    X_train_sel = X_train[:, indices]
    X_test_sel = X_test[:, indices]
    
    # Use Linear Regression for this medical regression task
    model = LinearRegression()
    model.fit(X_train_sel, y_train)
    
    predictions = model.predict(X_test_sel)
    
    # Calculate Error (MSE)
    mse = mean_squared_error(y_test, predictions)
    
    # INVERSE Error: We want to MINIMIZE MSE, so we MAXIMIZE (1 / MSE)
    # Added 1e-5 to avoid division by zero
    fitness = 1.0 / (mse + 1e-5)
    return fitness

# 4. Genetic Algorithm Loop
def ga_feature_selection():
    global population
    
    for gen in range(GENERATIONS):
        fitness_scores = [calculate_fitness_fs(ind) for ind in population]
        
        # Track the best MSE in this generation (Inverse of fitness)
        best_gen_fitness = max(fitness_scores)
        best_mse = (1.0 / best_gen_fitness) if best_gen_fitness > 0 else float('inf')
        
        print(f"Gen {gen+1}: Best MSE = {best_mse:.2f} (Lower is better)")
        
        # Roulette Wheel Selection
        total_fitness = sum(fitness_scores)
        if total_fitness == 0:
            probs = [1/len(fitness_scores)] * len(fitness_scores)
        else:
            probs = [f / total_fitness for f in fitness_scores]
            
        new_population = []
        
        while len(new_population) < POPULATION_SIZE:
            # Select Parents
            indices = np.random.choice(len(population), size=2, p=probs, replace=True)
            p1, p2 = population[indices[0]], population[indices[1]]
            
            # Crossover
            point = random.randint(1, NUM_FEATURES - 1)
            c1 = np.concatenate((p1[:point], p2[point:]))
            c2 = np.concatenate((p2[:point], p1[point:]))
            
            # Mutation
            for child in [c1, c2]:
                if random.random() < MUTATION_RATE:
                    idx = random.randint(0, NUM_FEATURES - 1)
                    child[idx] = 1 - child[idx]
                
                # Correction: If mutation resulted in all 0s, fix it
                if np.sum(child) == 0:
                     child[random.randint(0, NUM_FEATURES - 1)] = 1
                     
                new_population.append(child)
        
        population = np.array(new_population[:POPULATION_SIZE])

    # Final Result extraction
    final_fitness = [calculate_fitness_fs(ind) for ind in population]
    best_idx = np.argmax(final_fitness)
    return population[best_idx], final_fitness[best_idx]

# --- Execution ---
best_mask, best_fit = ga_feature_selection()
best_mse = 1.0 / best_fit

selected_features = [feature_names[i] for i, x in enumerate(best_mask) if x == 1]

print("\n--- Final Results (Medical Regression) ---")
print(f"Selected {len(selected_features)}/{NUM_FEATURES} features.")
print(f"Features: {selected_features}")
print(f"Final MSE: {best_mse:.2f}")

Dataset Loaded: Diabetes (Regression Task)
Total Features: 10 (['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6'])
Gen 1: Best MSE = 2745.99 (Lower is better)
Gen 2: Best MSE = 2745.99 (Lower is better)
Gen 3: Best MSE = 2745.99 (Lower is better)
Gen 4: Best MSE = 2736.53 (Lower is better)
Gen 5: Best MSE = 2788.15 (Lower is better)
Gen 6: Best MSE = 2764.37 (Lower is better)
Gen 7: Best MSE = 2767.90 (Lower is better)
Gen 8: Best MSE = 2745.99 (Lower is better)
Gen 9: Best MSE = 2776.61 (Lower is better)
Gen 10: Best MSE = 2763.86 (Lower is better)

--- Final Results (Medical Regression) ---
Selected 6/10 features.
Features: ['bmi', 's1', 's2', 's3', 's4', 's5']
Final MSE: 2772.12
